# 🗓️ 21일차 스터디 노트북 — 퀵 정렬

**오늘 범위**: 06-6 퀵 정렬 → 배열을 두 그룹으로 나누기(partition) → 재귀로 퀵 정렬 완성 → 비재귀 퀵 정렬(스택 활용)

## 난이도 태그
🟢 **기본** / 🟡 **표준** / 🔴 **심화**

## 유형 태그
**[손]** 손으로 추적 · **[빈칸]** 빈칸 채우기 · [예측] · [구현] · [디버깅] · [설명] · [실험]

## 오늘의 핵심 질문
> **"가장 빠른 정렬"이라는 퀵 정렬, 뭐가 그렇게 빠를까?**

핵심은 **분할 정복**이야. 15일차 하노이, 16일차 8퀸에서 배운 그 전략이 정렬에 적용된 거지.

## ⚠️ 오늘의 코드 리뷰 결과
- `quick_sort1.py` / `quick_sort1_verbose.py`: **완벽** (랜덤 1000회 실패 0, 교재 263p 출력과 정확히 일치) 👍
- `partition.py`: **들여쓰기 버그 1개 발견** — 3번 문제에서 다룰게

> 📁 아래 "부록" 셀을 **먼저 실행**해.

---

## 🔁 [Remind] 워밍업 — 분할 정복 복습

퀵 정렬은 **05장에서 배운 분할 정복 알고리즘**이야 (교재 260p). 먼저 그 개념부터 되짚자.

### R-1. 🟢 [설명] 분할 정복, 어디서 배웠지?

교재 260p: "퀵 정렬은 05장에서 살펴본 8퀸 문제와 같은 **분할 정복 알고리즘**이므로 재귀 호출을 사용하여 구현할 수 있습니다."

| | 큰 문제 | 어떻게 쪼개나 | 재귀 호출 |
|---|---|---|---|
| 하노이 (15일) | 원반 n개 옮기기 | 맨 아래 1개 + 나머지 그룹 | `move(no-1, ...)` |
| 8퀸 (16일) | 퀸 8개 배치 | 1열 배치 + 나머지 열 | `set(i+1)` |
| **퀵 정렬 (오늘)** | 배열 n개 정렬 | **①____** | **②____** |

*(①② 채우기)*

- 세 알고리즘의 **공통 전략**을 한 문장으로 말해봐.
- 15일차에 배운 **"믿음의 도약"**이 퀵 정렬에도 적용될까?

*(여기에 답 작성)*

### R-2. 🟡 [설명] 18일차 셰이커 정렬과 닮은 구조

퀵 정렬의 분할(partition) 코드를 보면 낯익은 게 있어:

```python
pl = left          # 왼쪽 커서
pr = right         # 오른쪽 커서
while pl <= pr:
    ...
    pl += 1
    pr -= 1
```

- 18일차 **셰이커 정렬**에서도 `left`, `right` 두 커서를 썼지. 어떤 점이 비슷해?
- 저번 코딩테스트의 **구명보트** 문제도 같은 틀이었어. 이 패턴을 뭐라고 부를 수 있을까?
- 다만 **결정적으로 다른 점**이 하나 있어. 셰이커는 커서가 만나면 끝나지만, 퀵 정렬은 만난 뒤에 **뭘 더** 하지?

*(여기에 답 작성)*

---
## 📦 부록 — 코드 모음

In [ ]:
from typing import MutableSequence
from collections import deque
import random

# ---------- 11일차 Stack (비재귀 퀵 정렬용) ----------
class Stack:
    def __init__(self, maxlen: int = 256) -> None:
        self.capacity = maxlen
        self.__stk = deque([], maxlen)
    def __len__(self): return len(self.__stk)
    def is_empty(self): return not self.__stk
    def is_full(self): return len(self.__stk) == self.__stk.maxlen
    def push(self, value): self.__stk.append(value)
    def pop(self): return self.__stk.pop()
    def peek(self): return self.__stk[-1]
    def to_list(self): return list(self.__stk)


# ---------- 실습 6-10: 배열 나누기 (교재 원본) ----------
def partition_book(a):
    '''배열을 나누어 출력 (교재 258p 원본)'''
    n = len(a)
    pl = 0
    pr = n - 1
    x = a[n // 2]

    while pl <= pr:
        while a[pl] < x: pl += 1
        while a[pr] > x: pr -= 1
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]
            pl += 1
            pr -= 1

    print(f'피벗은 {x}입니다.')
    print('피벗 이하인 그룹입니다.')
    print(*a[0 : pl])
    if pl > pr + 1:
        print('피벗과 일치하는 그룹입니다.')
        print(*a[pr + 1 : pl])
    print('피벗 이상인 그룹입니다.')      # ← if 밖! (교재 29~30행)
    print(*a[pr + 1 : n])


# ---------- 실습 6-11: 퀵 정렬 ----------
def qsort(a, left, right):
    '''a[left] ~ a[right]를 퀵 정렬'''
    pl = left
    pr = right
    x = a[(left + right) // 2]

    while pl <= pr:
        while a[pl] < x: pl += 1
        while a[pr] > x: pr -= 1
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]
            pl += 1
            pr -= 1

    if left < pr: qsort(a, left, pr)
    if pl < right: qsort(a, pl, right)

def quick_sort(a):
    if len(a) > 0:
        qsort(a, 0, len(a) - 1)
    return a


# ---------- 실습 6C-3: 나누는 과정 출력 ----------
def qsort_verbose(a, left, right):
    pl = left
    pr = right
    x = a[(left + right) // 2]
    print(f'a[{left}] ~ a[{right}]:', *a[left : right + 1])
    while pl <= pr:
        while a[pl] < x: pl += 1
        while a[pr] > x: pr -= 1
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]
            pl += 1
            pr -= 1
    if left < pr: qsort_verbose(a, left, pr)
    if pl < right: qsort_verbose(a, pl, right)


# ---------- 실습 6-12: 비재귀 퀵 정렬 ----------
def qsort_nonrec(a, left, right):
    '''스택을 사용한 비재귀 퀵 정렬'''
    rng = Stack(right - left + 1)
    rng.push((left, right))

    while not rng.is_empty():
        pl, pr = left, right = rng.pop()
        x = a[(left + right) // 2]

        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]
                pl += 1
                pr -= 1

        if left < pr: rng.push((left, pr))
        if pl < right: rng.push((pl, right))
    return a


def test_sort(f, trials=500):
    fails, example = 0, None
    for _ in range(trials):
        x = [random.randint(0, 50) for _ in range(random.randint(1, 15))]
        expected = sorted(x)
        y = x[:]
        f(y)
        if y != expected:
            fails += 1
            if example is None: example = (x, y, expected)
    return fails, example


def trace_partition(a):
    a = list(a)
    n = len(a)
    pl, pr = 0, n - 1
    x = a[n // 2]
    print(f'배열: {a}')
    print(f'피벗 x = a[{n//2}] = {x}')
    print()
    step = 1
    while pl <= pr:
        while a[pl] < x: pl += 1
        while a[pr] > x: pr -= 1
        print(f'단계 {step}: pl={pl}(값 {a[pl]}), pr={pr}(값 {a[pr]})', end='')
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]
            print(f' → 교환 → {a}')
            pl += 1
            pr -= 1
        else:
            print(' → 교차! (pl > pr) 나누기 종료')
        step += 1
    print()
    print(f'최종: {a}')
    print(f'  pl={pl}, pr={pr}')
    print(f'  피벗 이하 그룹: a[0:{pl}] = {a[0:pl]}')
    print(f'  피벗 이상 그룹: a[{pr+1}:{n}] = {a[pr+1:n]}')
    if pl > pr + 1:
        print(f'  피벗 일치 그룹: a[{pr+1}:{pl}] = {a[pr+1:pl]}')
    else:
        print(f'  (피벗 일치 그룹 없음: pl={pl} <= pr+1={pr+1})')

print('준비 완료')


---
## 📖 오늘의 핵심 개념

### 1. 퀵 정렬이란
찰스 A. R. 호어(Charles A. R. Hoare)가 고안. 정렬 속도가 매우 **빠르다(quick)**고 해서 붙은 이름.

**아이디어** (교재 255p 그림 6-19):
1. **피벗(pivot)**을 하나 고른다 (그룹을 나누는 기준값)
2. 피벗보다 작은 그룹 / 큰 그룹으로 나눈다
3. 각 그룹에서 또 피벗을 골라 나누기를 반복
4. 모든 그룹이 1개씩 남으면 정렬 완료

### 2. 배열을 두 그룹으로 나누기 (partition) 🎯

```python
pl = 0            # 왼쪽 커서
pr = n - 1        # 오른쪽 커서
x = a[n // 2]     # 피벗 (가운데 원소)

while pl <= pr:
    while a[pl] < x: pl += 1    # 피벗 이상인 원소를 찾을 때까지 →
    while a[pr] > x: pr -= 1    # 피벗 이하인 원소를 찾을 때까지 ←
    if pl <= pr:
        a[pl], a[pr] = a[pr], a[pl]   # 교환!
        pl += 1
        pr -= 1
```

**동작 원리**:
- `pl`은 왼쪽에서 오른쪽으로 가며 **"여기 있으면 안 되는 큰 값"**(피벗 이상)을 찾아
- `pr`은 오른쪽에서 왼쪽으로 가며 **"여기 있으면 안 되는 작은 값"**(피벗 이하)을 찾아
- 둘 다 찾으면 **서로 교환** → 큰 값은 오른쪽으로, 작은 값은 왼쪽으로 이동

### 3. 나누기가 끝난 뒤의 3개 그룹 (교재 257p)

`pl`과 `pr`이 **교차**(`pl > pr`)하면 나누기 완료:

| 그룹 | 범위 | 조건 |
|---|---|---|
| 피벗 **이하** | `a[0] ~ a[pl-1]` | 항상 생성 |
| 피벗 **일치** | `a[pr+1] ~ a[pl-1]` | **`pl > pr + 1`일 때만** |
| 피벗 **이상** | `a[pr+1] ~ a[n-1]` | 항상 생성 |

> ⚠️ "피벗과 일치하는 그룹"은 **조건부**로만 생성되지만, "피벗 이상인 그룹"은 **항상** 출력돼야 해. (3번 문제에서 이게 왜 중요한지 볼 거야)

### 4. 같은 원소를 교환하는 경우 (교재 257~258p)
`pl`과 `pr`이 **둘 다 피벗과 같은 위치**를 가리키면, 자기 자신과 교환하는 무의미한 동작이 일어나. 근데 교재는 그냥 두라고 해:

> "매번 체크하는 횟수보다 **1번만 같은 원소를 교환하는 것이 비용이 적게** 듭니다."

즉 "예외 처리하는 비용 > 가끔 낭비되는 1번의 교환 비용"이라는 실용적 판단이야.

### 5. 퀵 정렬 = 분할 + 재귀

```python
def qsort(a, left, right):
    ... (분할) ...
    if left < pr: qsort(a, left, pr)      # 왼쪽 그룹 다시 나누기
    if pl < right: qsort(a, pl, right)    # 오른쪽 그룹 다시 나누기
```

**왜 조건이 붙나**: 원소가 1개인 그룹은 더 나눌 필요가 없으니까 (교재 260p).
- `left < pr`: 왼쪽 그룹에 원소가 2개 이상
- `pl < right`: 오른쪽 그룹에 원소가 2개 이상
- 가운데 그룹(`a[pr+1]~a[pl-1]`)은 **이미 다 같은 값**이라 제외

### 6. 비재귀 퀵 정렬 (실습 6-12)
14일차에 `recur`를 스택으로 바꿨던 것처럼, 퀵 정렬도 재귀 없이 만들 수 있어:

```python
rng = Stack(right - left + 1)
rng.push((left, right))              # 처음 범위를 스택에 넣고

while not rng.is_empty():
    pl, pr = left, right = rng.pop() # 범위를 꺼내서
    ... (분할) ...
    if left < pr: rng.push((left, pr))    # 나눈 범위를 다시 스택에
    if pl < right: rng.push((pl, right))
```

**14일차와 다른 점**: 그때는 숫자 하나(`n`)를 쌓았는데, 여기는 **`(left, right)` 튜플**을 통째로 쌓아. "어디부터 어디까지 정렬해야 하는지"를 기억해야 하니까.

### 7. 복잡도와 안정성
- **시간 복잡도**: 평균 **O(n log n)** — 지금까지 배운 것 중 가장 빠름!
- **안정성**: ❌ **불안정** — 서로 이웃하지 않는 원소를 교환하므로 (19일차 선택 정렬, 20일차 셸 정렬과 같은 이유)

---
## 🟢 기본 문제

### 1. 🟢 [손] 분할 과정 손으로 추적 🔥

배열 `[5, 7, 1, 4, 6, 2, 3, 9, 8]`을 나눠봐. (교재 256p)
- 피벗 `x = a[9//2] = a[4] = 6`
- `pl = 0`, `pr = 8`

| 단계 | pl이 멈춘 위치 | pr이 멈춘 위치 | 교환 후 배열 |
|---|---|---|---|
| 1 | 인덱스 1 (값 7) | 인덱스 6 (값 3) | `5 3 1 4 6 2 7 9 8` |
| 2 | **①____** | **②____** | **③____** |
| 3 | 교차! | | 완료 |

*(①②③ 채우고 아래로 확인)*

In [ ]:
trace_partition([5, 7, 1, 4, 6, 2, 3, 9, 8])


### 2. 🟢 [예측] 피벗과 일치하는 그룹이 생기는 경우

교재 257p 그림 6-20의 예시 `[1, 8, 7, 4, 5, 2, 6, 3, 9]`를 나눠봐.

**실행 전 예측**:
- 피벗은 뭐야? (`a[9//2]`)
- 이번엔 "피벗과 일치하는 그룹"이 생길까?

In [ ]:
trace_partition([1, 8, 7, 4, 5, 2, 6, 3, 9])


**(a)** 1번(`[5,7,1,4,6,2,3,9,8]`)과 2번(`[1,8,7,4,5,2,6,3,9]`)의 차이가 뭐야? 왜 한쪽만 "일치 그룹"이 생겨?

**(b)** `pl > pr + 1`이 참이라는 건 정확히 무슨 뜻일까? (힌트: `pl`과 `pr`이 얼마나 벌어졌는지)

*(여기에 답 작성)*

---
### 3. 🟡 [디버깅] `partition.py`의 들여쓰기 버그 🔥🔥

네가 짠 `partition.py`는 이렇게 되어 있어:

```python
    if pl > pr + 1:
        print('피벗과 일치하는 그룹입니다.')
        print(*a[pr + 1 : pl])

        print('피벗 이상인 그룹입니다.')     # ← if 안에 들여쓰기됨!
        print(*a[pr + 1 : n])
```

교재 258p(25~30행)는 이래:
```python
    if pl > pr + 1:
        print('피벗과 일치하는 그룹입니다.')
        print(*a[pr + 1 : pl])

    print('피벗 이상인 그룹입니다.')          # ← if 밖!
    print(*a[pr + 1 : n])
```

직접 비교해봐.

In [ ]:
def partition_bug(a):
    '''네 코드: "피벗 이상인 그룹"이 if 안에 있음'''
    n = len(a); pl = 0; pr = n - 1; x = a[n // 2]
    while pl <= pr:
        while a[pl] < x: pl += 1
        while a[pr] > x: pr -= 1
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]; pl += 1; pr -= 1
    print(f'피벗은 {x}입니다.')
    print('피벗 이하인 그룹입니다.')
    print(*a[0 : pl])
    if pl > pr + 1:
        print('피벗과 일치하는 그룹입니다.')
        print(*a[pr + 1 : pl])
        print('피벗 이상인 그룹입니다.')      # ← if 안
        print(*a[pr + 1 : n])

print('=== 케이스 A: [1,8,7,4,5,2,6,3,9] (일치 그룹 생김) ===')
print('--- 네 코드 ---')
partition_bug([1,8,7,4,5,2,6,3,9])
print('--- 교재 ---')
partition_book([1,8,7,4,5,2,6,3,9])
print()
print('=== 케이스 B: [5,7,1,4,6,2,3,9,8] (일치 그룹 없음) ===')
print('--- 네 코드 ---')
partition_bug([5,7,1,4,6,2,3,9,8])
print('--- 교재 ---')
partition_book([5,7,1,4,6,2,3,9,8])


**(a)** 케이스 A에서는 두 코드의 출력이 같아. 케이스 B에서는 어떻게 달라?

**(b)** **왜** 케이스 A에서는 버그가 안 드러날까? 이게 왜 위험한 종류의 버그야?
*(힌트: 18일차 9번, 13일차 `factorial(-1)`, 9일차 오픈해시 `None` 버그와 같은 종류)*

**(c)** "피벗 이상인 그룹"은 **항상** 생성되는데 "피벗과 일치하는 그룹"은 **조건부**로만 생성돼. 이 논리적 차이가 코드 구조(들여쓰기)에 어떻게 반영되어야 할까?

*(여기에 답 작성)*

---
### 4. 🟢 [빈칸] 분할(partition) 구현

TODO를 채워봐. **부등호 방향**이 핵심이야.

In [ ]:
def my_partition(a):
    n = len(a)
    pl = 0
    pr = n - 1
    x = a[___]                      # TODO: 피벗 (가운데 원소)

    while pl <= pr:
        while a[pl] ___ x: pl += 1  # TODO: 피벗 이상인 원소를 찾을 때까지
        while a[pr] ___ x: pr -= 1  # TODO: 피벗 이하인 원소를 찾을 때까지
        if pl <= pr:
            a[pl], a[pr] = ___, ___ # TODO: 교환
            pl += 1
            pr -= 1
    return a, pl, pr

a, pl, pr = my_partition([5,7,1,4,6,2,3,9,8])
print(f'결과: {a}, pl={pl}, pr={pr}')
print(f'기대: [5, 3, 1, 4, 2, 6, 7, 9, 8], pl=5, pr=4')


---
## 🟡 표준 문제

### 5. 🟡 [손] 재귀 호출 흐름 추적 🔥

`quick_sort1_verbose.py`(실습 6C-3)로 `[5, 8, 4, 2, 6, 1, 3, 9, 7]`을 정렬해봐. (교재 263p)

**실행 전 예측**: 첫 번째 분할 후 왼쪽 그룹과 오른쪽 그룹의 범위는?

In [ ]:
x = [5, 8, 4, 2, 6, 1, 3, 9, 7]
print('=== 나누는 과정 ===')
qsort_verbose(x, 0, len(x) - 1)
print()
print(f'최종 정렬 결과: {x}')


**(a)** 출력을 보면 `a[0]~a[8]` 다음에 `a[0]~a[4]`가 나와. 왜 **왼쪽 그룹이 먼저** 처리될까? (코드의 어느 줄 때문인지)

**(b)** `a[0]~a[1]`까지 내려간 뒤 `a[3]~a[4]`로 올라와. 이 순서가 **13~14일차에 배운 재귀의 어떤 성질**과 관련 있어?

**(c)** 재귀 호출 트리를 그려봐. (15일차 하노이 10번처럼)

*(여기에 답 작성)*

---
### 6. 🟡 [설명] `if left < pr` / `if pl < right` 조건의 의미

```python
if left < pr: qsort(a, left, pr)
if pl < right: qsort(a, pl, right)
```

교재 260p: "원소 수가 1개인 그룹은 더 이상 나눌 필요가 없으므로 원소 수가 2개 이상인 그룹만 반복해서 나눕니다."

- `left < pr`이 거짓이면 왼쪽 그룹의 원소가 몇 개야?
- 왜 **가운데 그룹**(`a[pr+1] ~ a[pl-1]`)은 재귀 호출을 안 할까?
- 만약 이 조건들을 빼고 무조건 재귀 호출하면 어떻게 될까? (13일차 기저 조건 복습!)

*(여기에 답 작성)*

---
### 7. 🟡 [실험] 퀵 정렬은 안정적일까

19일차 선택 정렬, 20일차 셸 정렬처럼 확인해봐.

In [ ]:
def qsort_tagged(a, left, right):
    pl, pr = left, right
    x = a[(left + right) // 2][0]
    while pl <= pr:
        while a[pl][0] < x: pl += 1
        while a[pr][0] > x: pr -= 1
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]
            pl += 1; pr -= 1
    if left < pr: qsort_tagged(a, left, pr)
    if pl < right: qsort_tagged(a, pl, right)
    return a

tests = [
    [(2,'L'), (2,'R'), (1,'X'), (2,'M')],
    [(3,'a'), (1,'b'), (3,'c'), (2,'d'), (3,'e')],
]
for data in tests:
    y = data[:]
    qsort_tagged(y, 0, len(y)-1)
    stable = sorted(data, key=lambda t: t[0])
    print(f'입력       : {data}')
    print(f'퀵 정렬    : {y}')
    print(f'안정적이면 : {stable}')
    print(f'같은가? {y == stable}')
    print()


**(a)** 퀵 정렬은 안정적이야, 불안정해?

**(b)** **왜** 그럴까? 19일차 선택 정렬, 20일차 셸 정렬이 불안정했던 이유와 비교해봐.

**(c)** 지금까지 배운 정렬 중 **안정적인 것**과 **불안정한 것**을 각각 나열해봐. 공통점이 뭐야?

*(여기에 답 작성)*

---
### 8. 🟡 [실험] 얼마나 빠른가 — 5형제 총력전 🔥

지금까지 배운 정렬들의 연산 횟수를 비교해봐.

In [ ]:
import random

def bubble_cnt(a):
    n = len(a); c = 0
    for i in range(n-1):
        for j in range(n-1, i, -1):
            c += 1
            if a[j-1] > a[j]: a[j-1], a[j] = a[j], a[j-1]
    return c

def insertion_cnt(a):
    n = len(a); c = 0
    for i in range(1, n):
        j = i; tmp = a[i]
        while j > 0 and a[j-1] > tmp:
            c += 1; a[j] = a[j-1]; j -= 1
        a[j] = tmp
    return c

def shell_cnt(a):
    n = len(a); h = n//2; c = 0
    while h > 0:
        for i in range(h, n):
            j = i-h; tmp = a[i]
            while j >= 0 and a[j] > tmp:
                c += 1; a[j+h] = a[j]; j -= h
            a[j+h] = tmp
        h //= 2
    return c

def quick_cnt(a):
    cnt = [0]
    def q(a, l, r):
        pl, pr = l, r; x = a[(l+r)//2]
        while pl <= pr:
            while a[pl] < x: pl += 1; cnt[0] += 1
            while a[pr] > x: pr -= 1; cnt[0] += 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]; cnt[0] += 1
                pl += 1; pr -= 1
        if l < pr: q(a, l, pr)
        if pl < r: q(a, pl, r)
    if len(a) > 0: q(a, 0, len(a)-1)
    return cnt[0]

print(f'{"n":>6} {"버블":>10} {"삽입":>10} {"셸":>9} {"퀵":>8}')
print('-' * 48)
for n in [50, 100, 500, 1000]:
    t = [0]*4
    trials = 30
    for _ in range(trials):
        base = [random.randint(0, 10000) for _ in range(n)]
        t[0] += bubble_cnt(base[:]); t[1] += insertion_cnt(base[:])
        t[2] += shell_cnt(base[:]);  t[3] += quick_cnt(base[:])
    print(f'{n:>6} {t[0]/trials:>10.0f} {t[1]/trials:>10.0f} {t[2]/trials:>9.0f} {t[3]/trials:>8.0f}')

print()
import math
print(f'참고: n=1000일 때  n²={1000**2:,}  |  n·log₂n={1000*math.log2(1000):,.0f}')


**(a)** 퀵 정렬이 **O(n log n)**인데, 표에서 셸 정렬(O(n^1.25))보다 연산 횟수가 조금 많게 나와. 왜 그럴까?
*(힌트: 여기서 센 건 "비교+교환 연산 횟수"지 실제 실행 시간이 아니야. 또 n=1000은 아직 작은 편)*

**(b)** 그럼에도 퀵 정렬을 "가장 빠른 정렬"이라 부르는 이유는? (힌트: n이 훨씬 커지면?)

*(여기에 답 작성)*

---
## 🔴 심화 문제

### 9. 🔴 [실험] 피벗 선택이 왜 중요한가 🔥🔥

교재 259p: "피벗은 어떤 값으로 선택하느냐에 따라 배열을 나누는 것과 정렬하는 성능에 영향을 미칩니다."

**가운데 원소**를 피벗으로 쓸 때와 **맨 앞 원소**를 쓸 때의 재귀 깊이를 비교해봐.

In [ ]:
import math, sys

def qsort_depth_mid(a, left, right, d=0):
    '''가운데 원소를 피벗으로'''
    md = d
    pl, pr = left, right
    x = a[(left + right) // 2]
    while pl <= pr:
        while a[pl] < x: pl += 1
        while a[pr] > x: pr -= 1
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]; pl += 1; pr -= 1
    if left < pr: md = max(md, qsort_depth_mid(a, left, pr, d+1))
    if pl < right: md = max(md, qsort_depth_mid(a, pl, right, d+1))
    return md

def qsort_depth_first(a, left, right, d=0):
    '''맨 앞 원소를 피벗으로 (나쁜 선택)'''
    md = d
    pl, pr = left, right
    x = a[left]
    while pl <= pr:
        while a[pl] < x: pl += 1
        while a[pr] > x: pr -= 1
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]; pl += 1; pr -= 1
    if left < pr: md = max(md, qsort_depth_first(a, left, pr, d+1))
    if pl < right: md = max(md, qsort_depth_first(a, pl, right, d+1))
    return md

sys.setrecursionlimit(5000)
print(f'{"n":>5} {"가운데(랜덤)":>14} {"가운데(정렬됨)":>16} {"맨앞(정렬됨)":>14} {"log₂n":>8}')
print('-' * 62)
for n in [15, 31, 63, 127]:
    rnd = [random.randint(0, 1000) for _ in range(n)]
    srt = list(range(n))
    d1 = qsort_depth_mid(rnd[:], 0, n-1)
    d2 = qsort_depth_mid(srt[:], 0, n-1)
    d3 = qsort_depth_first(srt[:], 0, n-1)
    print(f'{n:>5} {d1:>14} {d2:>16} {d3:>14} {math.log2(n):>8.1f}')


**(a)** **맨 앞 원소를 피벗으로 + 이미 정렬된 배열**일 때 재귀 깊이가 거의 `n`에 육박해. 왜 그럴까?
*(힌트: 정렬된 배열에서 맨 앞이 최솟값이면, 나누기 결과 한쪽 그룹에 몇 개가 들어갈까?)*

**(b)** 이 상황에서 퀵 정렬의 복잡도는 O(n log n)이 아니라 **O(n²)**로 퇴화해. 왜 그런지 설명해봐.

**(c)** 교재가 **가운데 원소**를 피벗으로 고른 이유는? 이게 완벽한 해결책일까?

*(여기에 답 작성)*

---
### 10. 🔴 [빈칸] 비재귀 퀵 정렬 (실습 6-12)

14일차에 `recur`를 스택으로 바꿨던 것처럼, 퀵 정렬도 재귀 없이 만들어봐.

In [ ]:
def my_qsort_nonrec(a, left, right):
    rng = Stack(right - left + 1)
    rng.push((___, ___))                  # TODO: 처음 범위를 스택에

    while not rng.is_empty():
        pl, pr = left, right = rng.___()  # TODO: 범위를 꺼냄
        x = a[(left + right) // 2]

        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]
                pl += 1
                pr -= 1

        if left < pr: rng.___((left, pr))    # TODO: 왼쪽 그룹 저장
        if pl < right: rng.___((pl, right))  # TODO: 오른쪽 그룹 저장
    return a

x = [5,8,4,2,6,1,3,9,7]
my_qsort_nonrec(x, 0, len(x)-1)
print(f'결과: {x}')

def wrapper(a):
    if len(a) > 0: my_qsort_nonrec(a, 0, len(a)-1)
fails, ex = test_sort(wrapper, 300)
print(f'랜덤 300회 실패: {fails}')


**(a)** 14일차 비재귀 `recur`에서는 스택에 **숫자 하나**(`n`)를 쌓았는데, 여기는 **튜플 `(left, right)`**를 쌓아. 왜 두 개가 필요할까?

**(b)** 스택 크기를 `right - left + 1`(= 배열 크기)로 잡았어. 이게 충분할까? (11일차 `FixedStack` vs `deque(maxlen)` 논의 회수)

**(c)** 재귀 버전과 비재귀 버전의 **처리 순서**가 다를까? 아래 실험으로 확인해봐.

In [ ]:
def qsort_order_rec(a, left, right, log):
    log.append((left, right))
    pl, pr = left, right
    x = a[(left + right) // 2]
    while pl <= pr:
        while a[pl] < x: pl += 1
        while a[pr] > x: pr -= 1
        if pl <= pr:
            a[pl], a[pr] = a[pr], a[pl]; pl += 1; pr -= 1
    if left < pr: qsort_order_rec(a, left, pr, log)
    if pl < right: qsort_order_rec(a, pl, right, log)

def qsort_order_nonrec(a, left, right, log):
    rng = Stack(right - left + 1)
    rng.push((left, right))
    while not rng.is_empty():
        pl, pr = left, right = rng.pop()
        log.append((left, right))
        x = a[(left + right) // 2]
        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr:
                a[pl], a[pr] = a[pr], a[pl]; pl += 1; pr -= 1
        if left < pr: rng.push((left, pr))
        if pl < right: rng.push((pl, right))

base = [5,8,4,2,6,1,3,9,7]
log1, log2 = [], []
qsort_order_rec(base[:], 0, 8, log1)
qsort_order_nonrec(base[:], 0, 8, log2)
print('재귀   처리 순서:', log1)
print('비재귀 처리 순서:', log2)
print(f'같은가? {log1 == log2}')


### 11. 🔴 [디버깅] 빈 배열을 넣으면?

`quick_sort1.py`의 `quick_sort` 함수:
```python
def quick_sort(a):
    qsort(a, 0, len(a) - 1)
```

빈 배열 `[]`을 넣으면 어떻게 될까?

In [ ]:
def quick_sort_orig(a):
    '''교재 원본'''
    qsort(a, 0, len(a) - 1)

print('=== 빈 배열 테스트 ===')
try:
    quick_sort_orig([])
    print('  성공')
except Exception as e:
    print(f'  {type(e).__name__}: {e}')

print()
print('=== 원소 1개 ===')
try:
    y = [42]
    quick_sort_orig(y)
    print(f'  성공: {y}')
except Exception as e:
    print(f'  {type(e).__name__}: {e}')


**(a)** 빈 배열에서 무슨 에러가 났어? `len([]) - 1`은 뭐가 되지?

**(b)** 어떻게 고치면 될까?

**(c)** 원소가 1개일 때는 왜 문제가 없을까? (`qsort(a, 0, 0)`이 어떻게 동작하는지 추적해봐)

*(여기에 답 작성)*

---
### 12. 🔴 [종합] 정렬 알고리즘 6형제 총정리

| 정렬 | 시간 복잡도 | 안정성 | 핵심 아이디어 | 일차 |
|---|---|---|---|---|
| 버블 | O(n²) | ✅ | 이웃끼리 비교·교환 | 18일 |
| 셰이커 | O(n²) | ✅ | 양방향 스캔 | 18일 |
| 선택 | O(n²) | ❌ | 최솟값 선택 후 교환 | 19일 |
| 삽입 | O(n²) | ✅ | 알맞은 위치에 삽입 | 19일 |
| 셸 | O(n^1.25) | ❌ | h칸 간격으로 미리 정렬 | 20일 |
| **퀵** | **①____** | **②____** | **③____** | 21일 |

**(a)** ①②③ 채우기

**(b)** **안정적인 정렬**(버블·셰이커·삽입)과 **불안정한 정렬**(선택·셸·퀵)의 공통점을 각각 찾아봐.

**(c)** 실무에서 "그냥 `sorted()` 쓰면 되는데 왜 이걸 다 배워?"라는 질문에 뭐라고 답할래?

*(여기에 답 작성)*

---
## ✅ 정답 & 해설

### R-1. 분할 정복
① **피벗 기준으로 두 그룹(작은 값 / 큰 값)으로 나눔**
② **`qsort(a, left, pr)` / `qsort(a, pl, right)`**

- **공통 전략**: "큰 문제를 **같은 모양의 더 작은 문제**로 쪼개고, 각각에 **똑같은 함수를 재귀 호출**해서 해결한다"
- **믿음의 도약 적용됨** ✅: `qsort(a, left, pr)`를 호출할 때, 그 안에서 어떻게 정렬되는지 **몰라도** "왼쪽 그룹은 알아서 정렬될 것"이라 믿고 짜면 돼. 우리가 신경 쓸 건 **"이번 단계에서 두 그룹으로 잘 나누는 것"**뿐이야.

---
### R-2. 셰이커 정렬과의 비교
- **비슷한 점**: 양쪽 끝에서 커서 두 개(`pl`, `pr`)가 **서로를 향해 좁혀오는** 구조 (투 포인터)
- **패턴 이름**: **투 포인터(two pointer)**. 구명보트 문제도 같은 틀
- **결정적 차이**: 셰이커는 커서가 만나면 **정렬이 끝나**. 퀵 정렬은 커서가 교차하면 **"나누기"만 끝난 거고, 나눠진 두 그룹을 다시 정렬해야 해** → 그래서 **재귀 호출**이 필요!

---
### 1. 분할 과정 추적
① 인덱스 4 (값 6) / ② 인덱스 5 (값 2) / ③ `5 3 1 4 2 6 7 9 8`

**최종**: `pl=5, pr=4` (교차!)
- 피벗 이하 그룹: `[5, 3, 1, 4, 2]`
- 피벗 이상 그룹: `[6, 7, 9, 8]`

---
### 2. 피벗 일치 그룹
**(a)** `[1,8,7,4,5,2,6,3,9]`은 피벗이 `5`인데, 나누는 과정에서 `pl`과 `pr`이 **둘 다 피벗과 같은 값**(5)을 가리키는 순간이 생겨. 이때 "같은 원소끼리 교환"이 일어나면서 `pl`과 `pr`이 **2칸 이상 벌어져** → `pl > pr + 1` 성립.

`[5,7,1,4,6,2,3,9,8]`은 그런 순간이 없어서 `pl = pr + 1`(딱 1칸 차이)로 끝나 → 일치 그룹 없음.

**(b)** `pl > pr + 1`은 "`pl`과 `pr` 사이에 **최소 1개 이상의 원소**가 끼어 있다"는 뜻이야. 그 사이 원소들(`a[pr+1] ~ a[pl-1]`)이 바로 **피벗과 같은 값**들이야.

---
### 3. 들여쓰기 버그 🔥🔥
**(a)** 케이스 A(일치 그룹 있음)에서는 두 출력이 **같아**. 케이스 B(일치 그룹 없음)에서는 네 코드가 **"피벗 이상인 그룹"을 아예 출력 안 해!**

**(b)** 케이스 A에서는 `if pl > pr + 1`이 **참**이라 if 블록 전체가 실행돼서 버그가 안 드러나. 조건이 거짓일 때만 문제가 생기니, **특정 입력에서만 틀리는 조용한 버그**야.
- 18일차 9번(`if exchng == 0` 위치)
- 13일차 `factorial(-1)`이 조용히 1 반환
- 9일차 오픈해시가 `None` 값을 "키 없음"으로 착각

전부 같은 종류 — **"어떤 입력에서는 우연히 맞아서 발견이 늦어지는" 버그**. 그래서 **여러 입력으로 테스트하는 게 중요**해.

**(c)** 코드 구조가 **논리를 그대로 반영**해야 해:
- "피벗 이상 그룹"은 **항상** 생성 → `if` **밖**에 (들여쓰기 없음)
- "피벗 일치 그룹"은 **조건부** 생성 → `if` **안**에 (들여쓰기 있음)

파이썬에서 **들여쓰기가 곧 논리 구조**니까, 들여쓰기를 잘못하면 의도한 논리와 실제 동작이 어긋나.

---
### 4. 분할 구현 정답
```python
x = a[n // 2]
while a[pl] < x: pl += 1      # 피벗보다 작으면 계속 전진 → 멈추면 피벗 이상
while a[pr] > x: pr -= 1      # 피벗보다 크면 계속 후진 → 멈추면 피벗 이하
a[pl], a[pr] = a[pr], a[pl]
```
**부등호 주의**: `<`와 `>`(등호 없음)를 써야 **피벗과 같은 값에서 멈춰서** 교환이 일어나. `<=`, `>=`로 하면 커서가 피벗을 지나쳐 무한 루프나 범위 초과가 날 수 있어.

---
### 5. 재귀 흐름
**(a)** 코드 순서 때문이야:
```python
if left < pr: qsort(a, left, pr)      # ← 이게 먼저
if pl < right: qsort(a, pl, right)
```
왼쪽 그룹 호출이 **먼저 쓰여 있으니**, 그게 **완전히 끝날 때까지** 오른쪽 호출은 시작도 안 해.

**(b)** **13~14일차 콜 스택(LIFO)** 성질이야. `qsort(a,0,4)`를 호출하면 그 안에서 또 `qsort(a,0,2)`, `qsort(a,0,1)`로 계속 **깊이 내려가고**, 바닥에 닿아야 되돌아 나와서 형제 호출(`a[3]~a[4]`)로 넘어가. 15일차 하노이의 "내려가기 → 올라오기"와 완전히 같아.

**(c)** 재귀 호출 트리:
```
a[0]~a[8]
├─ a[0]~a[4]
│  ├─ a[0]~a[2]
│  │  └─ a[0]~a[1]
│  └─ a[3]~a[4]
└─ a[5]~a[8]
   ├─ a[5]~a[6]
   └─ a[7]~a[8]
```

---
### 6. 재귀 조건의 의미
- `left < pr`이 거짓 = `left >= pr` → 왼쪽 그룹의 원소가 **1개 이하**. 1개짜리는 이미 정렬된 것이니 나눌 필요 없음.
- **가운데 그룹**(`a[pr+1] ~ a[pl-1]`)은 전부 **피벗과 같은 값**이야. 이미 다 같은 값이니 **정렬할 게 없어** → 재귀 제외 (교재 260p 각주)
- 조건 없이 무조건 재귀하면 원소 1개짜리도 계속 호출해서 **무한 재귀 → RecursionError**. 13일차에 배운 **기저 조건**이 없는 셈!

---
### 7. 퀵 정렬의 안정성
**(a)** **불안정(unstable)** ❌ — 두 케이스 모두 안정적 정렬 결과와 달라.

**(b)** 퀵 정렬은 `a[pl]`과 `a[pr]`을 교환하는데, 이 둘은 **멀리 떨어져 있을 수 있어.** 교환 한 번으로 중간의 같은 값들을 **건너뛰어** 자리를 옮기니 순서가 깨져.
- 19일차 **선택 정렬**: 최솟값을 맨 앞과 교환 → 멀리 떨어진 교환
- 20일차 **셸 정렬**: h칸 떨어진 원소 교환 → 멀리 떨어진 교환
- 21일차 **퀵 정렬**: `pl`, `pr` 위치 교환 → 멀리 떨어진 교환

**전부 같은 원리!**

**(c)**
- **안정적**: 버블, 셰이커, 삽입 → **공통점: 이웃한(인접한) 원소만 교환/이동**
- **불안정**: 선택, 셸, 퀵 → **공통점: 멀리 떨어진 원소를 교환**

---
### 8. 성능 비교
**(a)** 두 가지 이유:
1. 여기서 센 건 **비교+교환 "횟수"**지 실제 **실행 시간**이 아니야. 퀵 정렬은 연산 하나하나가 단순해서 실제로는 더 빠를 수 있어.
2. **n=1000은 아직 작아.** O(n log n)과 O(n^1.25)의 차이는 n이 훨씬 커져야 확실히 드러나. (n=1000일 때 n·log₂n ≈ 9,966이고 n^1.25 ≈ 5,623이라 실제로 셸이 유리한 구간)

**(b)** n이 아주 커지면 **O(n log n)이 O(n^1.25)를 확실히 앞질러.** 또 퀵 정렬은 **캐시 지역성이 좋고**(연속된 메모리를 순차 접근), 상수 계수가 작아서 실제 구현에서 매우 빨라. 그래서 대부분의 표준 라이브러리가 퀵 정렬 계열을 기반으로 써.

---
### 9. 피벗 선택 🔥🔥
**결과**: n=127일 때 **가운데 피벗 = 깊이 5~9**, **맨 앞 피벗 + 정렬된 배열 = 깊이 125!**

**(a)** 이미 정렬된 배열에서 맨 앞이 **최솟값**이야. 그걸 피벗으로 쓰면 "피벗보다 작은 그룹"에 **아무것도 안 들어가고**, "피벗 이상 그룹"에 **나머지 n-1개가 전부** 들어가. 즉 **매번 1개씩만 떼어내는** 꼴이라 재귀가 n번 깊어져.

**(b)** 매 단계에서 문제 크기가 **절반**으로 줄어야 `log n` 깊이가 나와. 그런데 **1개씩만** 줄어들면 깊이가 `n`이 되고, 각 단계에서 O(n) 비교를 하니 총 **O(n²)**로 퇴화. 게다가 재귀 깊이가 n이라 **RecursionError** 위험도 커져.

**(c)** **가운데 원소**는 "이미 정렬된 배열"이나 "역순 배열" 같은 **흔한 최악 케이스를 피할 수 있어.** 정렬된 배열의 가운데는 대략 중앙값이니 그룹이 반반으로 잘 나뉘거든.

다만 **완벽하진 않아** — 가운데가 최솟값/최댓값이 되도록 **일부러 만든 입력**에는 여전히 취약해. 실무에서는 **median-of-three**(맨앞·가운데·맨뒤 세 값의 중앙값)나 **랜덤 피벗**을 써서 더 방어해.

---
### 10. 비재귀 퀵 정렬
```python
rng.push((left, right))
pl, pr = left, right = rng.pop()
if left < pr: rng.push((left, pr))
if pl < right: rng.push((pl, right))
```

**(a)** 14일차 `recur`는 "n부터 시작해서 아래로"라는 **한 방향 진행**이라 숫자 하나면 충분했어. 퀵 정렬은 "**배열의 어느 구간**을 정렬할지"를 기억해야 하는데, 구간은 **시작과 끝 두 개**로 표현되니까 `(left, right)` 튜플이 필요해.

**(b)** 충분해. 스택에 동시에 쌓이는 범위의 개수는 재귀 깊이와 같은데, 최악의 경우에도 배열 크기를 넘지 않아. 다만 **11일차 논의**대로 `deque(maxlen)`은 넘치면 **조용히 밀어내니**, `FixedStack`처럼 `Full` 예외를 던지는 편이 안전해. (교재 265p도 "스택의 크기"를 따로 다뤄)

**(c)** **처리 순서가 달라!** 재귀는 왼쪽 그룹을 먼저 끝까지 파고들지만(깊이 우선), 비재귀는 스택 LIFO 특성상 **마지막에 push한 오른쪽 그룹을 먼저 pop**해서 처리해. 결과는 같지만 **중간 과정의 순서**가 다르지.

---
### 11. 빈 배열
**(a)** `IndexError: list index out of range`. `len([]) - 1 = -1`이라 `qsort(a, 0, -1)`이 호출되고, 그 안에서 `x = a[(0 + -1) // 2] = a[-1]`인데 빈 배열엔 `a[-1]`이 없어서 터져.

**(b)** 가드를 추가:
```python
def quick_sort(a):
    if len(a) > 0:
        qsort(a, 0, len(a) - 1)
```

**(c)** 원소 1개면 `qsort(a, 0, 0)`이 호출돼. 그럼 `pl=0, pr=0, x=a[0]`. `while pl <= pr`에서 `a[0] < x`도 `a[0] > x`도 거짓(자기 자신이니 같음)이라 교환 후 `pl=1, pr=-1`이 돼. `left < pr`(0 < -1)도 `pl < right`(1 < 0)도 거짓이라 **재귀 없이 바로 종료** → 정상.

---
### 12. 6형제 총정리
**(a)** ① **O(n log n)** (평균) / ② **❌ 불안정** / ③ **피벗 기준 분할 + 재귀 (분할 정복)**

**(b)**
- **안정적**(버블·셰이커·삽입): 전부 **이웃한 원소끼리만** 교환하거나 한 칸씩 이동 → 같은 값이 서로를 앞지를 수 없음
- **불안정**(선택·셸·퀵): 전부 **멀리 떨어진 원소를 교환** → 중간의 같은 값을 건너뛰어 순서가 깨짐

**(c)** 몇 가지 답이 가능해:
1. **상황에 맞는 선택**: 거의 정렬된 데이터면 삽입 정렬이 더 빠르고, 안정성이 필요하면 불안정한 정렬을 쓰면 안 돼
2. **알고리즘적 사고**: 분할 정복, 재귀, 투 포인터, 백트래킹 같은 **범용 전략**을 배우는 것 — 정렬은 그 교보재일 뿐
3. **`sorted()`의 정체를 이해**: 파이썬의 `sorted()`는 **Timsort**(병합 정렬 + 삽입 정렬 혼합)야. 삽입 정렬의 특성을 알아야 왜 그렇게 설계됐는지 이해할 수 있어
4. **면접·코테**: 복잡도 분석과 트레이드오프를 설명할 수 있어야 함

---

## 📌 핵심 3줄 요약

1. **퀵 정렬 = 분할(partition) + 재귀**. 피벗을 기준으로 `pl`, `pr` 두 커서가 양쪽에서 좁혀오며 교환하고, 커서가 교차하면 나뉜 두 그룹을 **재귀로 다시 정렬**한다 (15~16일차 분할 정복!).
2. 평균 **O(n log n)**으로 가장 빠르지만, **피벗을 잘못 고르면 O(n²)로 퇴화**한다 (정렬된 배열 + 맨 앞 피벗 = 재귀 깊이 125). 그래서 교재는 **가운데 원소**를 피벗으로 쓴다.
3. 멀리 떨어진 원소를 교환하므로 **불안정**하다 — 선택 정렬(19일), 셸 정렬(20일)과 같은 이유.

## 🗂️ 스터디 진행 가이드
- 🟢 (1~4번): 전원 필수. **1번 분할 손추적**이 오늘의 기본기
- 🟡 (5~8번): 팀 목표선
  - **3번 들여쓰기 버그**는 네 코드에서 실제로 나온 것 🔥
  - 7번 불안정성, 8번 성능 비교
- 🔴 (9~12번): 도전
  - **9번 피벗 선택**이 오늘의 최고 문제 🔥🔥 — 깊이 5 vs 125의 극적 차이
  - 10번 비재귀(14일차 회수), 11번 빈 배열 엣지케이스
- **금요일 코딩테스트 범위**: 🟢🟡 (1~8번)

## 🔗 오늘 회수된 개념들
- **15일차 하노이 / 16일차 8퀸 분할 정복** → 퀵 정렬의 재귀 구조 (R-1, 5번)
- **13일차 기저 조건** → `if left < pr` 조건이 없으면 무한 재귀 (6번)
- **14일차 재귀 → 스택 변환** → 비재귀 퀵 정렬 (10번)
- **18일차 셰이커 / 구명보트 투 포인터** → `pl`, `pr` 커서 구조 (R-2)
- **19일차 선택 정렬 / 20일차 셸 정렬 불안정성** → 퀵도 같은 이유로 불안정 (7번)
- **18일차 9번 들여쓰기 버그** → `partition.py`도 같은 종류 (3번)
- **11일차 Stack** → 비재귀 버전에 그대로 재사용 (10번)

> **다음 진도**: 06-7 병합 정렬 → 06-8 힙 정렬 → 06-9 도수 정렬
